# 010 导数积分与局部变化

区分数学导数、有限差分与浮点实现。先读讲义，不能从几次数值结果证明极限存在。

In [1]:
from pathlib import Path
import csv
import experiment as lab
from test_experiment import run_tests
print("平方前向差分", lab.forward_difference(lab.square, 2, 0.1))
print("平方中心差分", lab.central_difference(lab.square, 2, 0.1))
print("解析导数", lab.analytic_derivative("square", 2))

平方前向差分 4.100000000000001
平方中心差分 4.000000000000001
解析导数 4


## 切线漏掉了什么
平方函数的有限增量误差为δ²。

In [2]:
for delta in [0.1, 0.5]:
    exact = lab.square(2 + delta)
    approximation = 4 + 4 * delta
    print(delta, exact, approximation, exact - approximation)
    assert abs((exact - approximation) - delta * delta) < 1e-12

0.1 4.41 4.4 0.009999999999999787
0.5 6.25 6.0 0.25


## 尖点是重要反例
|x|在0处有值，但左右差商不同。

In [3]:
h = 0.001
right = (abs(h) - abs(0)) / h
left = (abs(0) - abs(-h)) / h
center = lab.central_difference(abs, 0, h)
print("右、左、中心", right, left, center)
try:
    lab.analytic_derivative("abs", 0)
except ValueError:
    print("两侧导数不存在；中心差分0不能改变这一点")

右、左、中心 1.0 -1.0 0.0
两侧导数不存在；中心差分0不能改变这一点


## 扫描步长并保留失败状态
特别小的h不自动更准确。

In [4]:
result = lab.run(Path.cwd() / "notebook_outputs")
with (Path.cwd() / "notebook_outputs/difference_scan.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    print(row["h"], row["forward_error"], row["central_error"], row["status"])

0.1 0.10412950274943134 0.0033579330393158635 computed
0.01 0.010102410156916708 3.356271293863955e-05 computed
0.001 0.0010072120630053583 3.356254461017727e-07 computed
0.0001 0.00010069098879883498 3.3550127120918205e-09 computed
1e-05 1.0068750196712273e-05 2.6564084265601196e-11 computed
1e-06 1.00653255374894e-06 4.359623773098065e-12 computed
1e-07 9.703785197601178e-08 6.617741910019959e-10 computed
1e-08 1.2660902104499883e-08 1.2660902104499883e-08 computed
1e-09 4.3142830774556273e-07 2.0938370282053143e-07 computed
1e-10 3.540052776696001e-06 1.319606727445688e-06 computed
1e-11 3.0185405367699758e-05 7.980944875196627e-06 computed
1e-12 0.00025223001029273107 3.0185405367699758e-05 computed
1e-13 0.0020285868496929815 0.00019185919955733155 computed
1e-14 0.01535126314519486 0.006853197347308271 computed
1e-15 0.2373958680702264 0.015351263145195082 computed
1e-16 2.0137527074704766 0.20669334177983645 computed
1e-17   unresolvable_input_step


## 四段累计与方向
先核对线性流量的3、5、4、4，再看平方函数。

In [5]:
for method in ["left", "right", "midpoint", "trapezoid"]:
    print(method, lab.integrate(lab.flow, 0, 2, 4, method))
print("平方中点", lab.integrate(lab.square, 0, 2, 4, "midpoint"))
print("平方梯形", lab.integrate(lab.square, 0, 2, 4, "trapezoid"))
print("平方精确积分", 8 / 3)
print("反向累计", lab.integrate(lab.flow, 2, 0, 4))

left 3.0
right 5.0
midpoint 4.0
trapezoid 4.0
平方中点 2.625
平方梯形 2.75
平方精确积分 2.6666666666666665
反向累计 -4.0


## 相同局部方向配不同步长
别把有限步长下降当成导数定义的自动结论。

In [6]:
for step in [0.1, 0.5, 1, 1.1]:
    print(lab.parameter_step(0, step))
assert lab.parameter_step(0, 1.1)["new_loss"] > 9

{'step': 0.1, 'old_w': 0, 'derivative': -6, 'new_w': 0.6000000000000001, 'old_loss': 9, 'new_loss': 5.76}
{'step': 0.5, 'old_w': 0, 'derivative': -6, 'new_w': 3.0, 'old_loss': 9, 'new_loss': 0.0}
{'step': 1, 'old_w': 0, 'derivative': -6, 'new_w': 6, 'old_loss': 9, 'new_loss': 9}
{'step': 1.1, 'old_w': 0, 'derivative': -6, 'new_w': 6.6000000000000005, 'old_loss': 9, 'new_loss': 12.960000000000004}


## 全套检查
包括Fraction和70位Decimal参照，不以待测函数自己生成预期。

In [7]:
print(run_tests())

{'status': 'passed', 'test_groups': ['exact square and cube difference expansions', 'smooth derivative comparisons and 70-digit Decimal exponential reference', 'cusp counterexample distinguishes a difference formula from a derivative', 'step positivity, resolvability, denominator, domain and correctly classified overflow checks', 'independent Fraction rectangle sums, trapezoids and oriented bounds', 'panel-count failures and signed versus absolute area', 'exact parameter-step arithmetic and finite-step overshoot', 'complete repeated run is byte-identical']}


## 重启并全部执行
报告一个无导数但中心差分为0的例子，一个浮点步长失效点，以及一个过大步长使损失上升的例子。